# Frost API

## Sett opp API-nøkkel

In [1]:
CLIENT_ID = "0fa358f8-1ba2-4c05-bff7-2977868375c2"

## Definer Hartevatn-koordinater

In [2]:
timenes_coords = (58.17792, 8.11984)

## Hent alle stasjoner og filtrer på koordinater

In [3]:
import requests
import pandas as pd
from geopy.distance import geodesic

# Hent alle stasjoner fra Frost
url_stations = "https://frost.met.no/sources/v0.jsonld"
response = requests.get(url_stations, auth=(CLIENT_ID,''))
stations = response.json()['data']

# Lag liste med stasjoner som har koordinater, og regn avstand til Timenes
stations_list = []
for s in stations:
    if 'geometry' in s and 'coordinates' in s['geometry']:
        coords = (s['geometry']['coordinates'][1], s['geometry']['coordinates'][0])  # [lat, lon]
        distance = geodesic(timenes_coords, coords).km
        stations_list.append({
            'id': s['id'],
            'name': s['name'],
            'distance_km': distance
        })

stations_df = pd.DataFrame(stations_list)
stations_df = stations_df.sort_values('distance_km')
stations_df.head(5)  # viser de 5 nærmeste stasjonene



,id,name,distance_km
106,SN39030,KRISTIANSAND - HÅNES,1.709396
1449,SN39040,KJEVIK,3.533726
1806,SN38250,E18 STUDEVANN,4.599112
1932,SN39150,KRISTIANSAND - SØMSKLEIVA,5.047591
1686,SN39010,KRISTIANSAND - FIDJEÅSEN,5.517336


In [4]:
# Velg nærmeste stasjon
nearest_station = stations_df.iloc[1]['id']
print("Henter data fra stasjon:", nearest_station)

Henter data fra stasjon: SN39040


## Sett ønsket periode og velg stasjon

In [5]:
vinter_periods = [
    ("2024-11-03T23:00:00Z", "2024-11-30T23:00:00Z"),
    ("2024-12-01T00:00:00Z", "2024-12-31T23:00:00Z"),
    ("2025-01-01T00:00:00Z", "2025-01-31T23:00:00Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:00:00Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:00:00Z"),
    ("2026-01-01T00:00:00Z", "2026-01-17T22:00:00Z")
]

Hent data fra Frost API

In [6]:
def fetch_weather(stasjon_id, start, end):

    elements = "air_temperature,sum(precipitation_amount PT1H)"

    url = f"https://frost.met.no/observations/v0.jsonld?sources={stasjon_id}&elements={elements}&referencetime={start}/{end}"

    response = requests.get(url, auth=(CLIENT_ID, ''))
    
    if response.status_code != 200:
        print("Feil:", response.status_code)
        return pd.DataFrame()
    
    data = response.json()
    observations = data.get('data', [])
    
    records = []

    for obs in observations:
        row = {"timestamp": obs.get('referenceTime')}
        
        for element in obs.get('observations', []):
            row[element['elementId']] = element.get('value')
        
        records.append(row)

    return pd.DataFrame(records)

Hent vinterperioder

In [7]:
def fetch_weather_periods_hourly(stasjon_id, periods):
    dfs = []

    for start, end in periods:
        df = fetch_weather(stasjon_id, start, end)
        dfs.append(df)

    if dfs:
        weather_df = pd.concat(dfs, ignore_index=True)

        weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])

        weather_df_hourly = weather_df[weather_df['timestamp'].dt.minute == 0].copy()

        weather_df_hourly = weather_df_hourly.sort_values('timestamp').reset_index(drop=True)

        return weather_df_hourly
    
    else:
        return pd.DataFrame()

Hent data fra vintermånedene

In [8]:
weather_df_hourly = fetch_weather_periods_hourly(nearest_station, vinter_periods)
weather_df_hourly.head()

,timestamp,air_temperature,sum(precipitation_amount PT1H)
0,2024-11-03 23:00:00+00:00,6.1,0.0
1,2024-11-04 00:00:00+00:00,4.9,0.0
2,2024-11-04 01:00:00+00:00,4.8,0.0
3,2024-11-04 02:00:00+00:00,4.8,0.0
4,2024-11-04 03:00:00+00:00,4.8,0.0


## Tidsformat

In [9]:
# Konverter til datetime hvis ikke gjort
weather_df_hourly['timestamp'] = pd.to_datetime(weather_df_hourly['timestamp'])

# Formater som '2025-01-18 09:00:00.0000000'
weather_df_hourly['timestamp'] = weather_df_hourly['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S.%f0')

weather_df_hourly.head()

,timestamp,air_temperature,sum(precipitation_amount PT1H)
0,2024-11-03 23:00:00.0000000,6.1,0.0
1,2024-11-04 00:00:00.0000000,4.9,0.0
2,2024-11-04 01:00:00.0000000,4.8,0.0
3,2024-11-04 02:00:00.0000000,4.8,0.0
4,2024-11-04 03:00:00.0000000,4.8,0.0


## Lagre data til Azure Blob

In [10]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import io

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_name = "vaerdata"
blob_name = "Timenes_weather.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

csv_buffer = io.StringIO()
weather_df_hourly.to_csv(csv_buffer, index=False)

blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

print("CSV lagret i Azure Blob Storage!")

CSV lagret i Azure Blob Storage!
